<a href="https://colab.research.google.com/github/MuhammadOkasha004/flyrank-ml-internship-work/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadOkasha004/flyrank-ml-internship-work/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Step 1: Data Contract — Unit of Analysis & Time Window

#### Contract Statement (5 Answers):
1. **Grain (One Row):** $1\text{ Row} = 1\text{ Pseudonymized Web Page (Content Item)}$ represented by a unique `content_hash_id` aggregated over a monthly decision window $T_0$.
2. **Table(s) Used:** Primary daily performance table `fact_content_daily_performance` joined with `dim_content` metadata via `content_hash_id`.
3. **Time Window:** Mid-Panel Month **`2026-03`** (March 1, 2026 to March 31, 2026).
4. **Target / Label:** Binary traffic decay proxy (`gsc_clicks < 30`) and continuous Decay Risk Score for page refresh prioritization.
5. **Deliberate Exclusion:** Future outcome data (Months 4–6, i.e., April–June 2026 logs) to strictly avoid Data Leakage.

#### Verification Findings:
* **Raw Daily Rows:** ~9.84 Million daily log records in March 2026.
* **Unique Content Items:** Exactly **331,437 unique web pages**.
* **Date Span:** Full calendar month from `2026-03-01` to `2026-03-31`.

In [ ]:
import duckdb
from huggingface_hub import get_token

# 1. Token Setup
token = get_token()
# Agar token None return kare, toh neeche wali line se comment (#) hatayein:
# token = "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"

# 2. Connection Setup
con = duckdb.connect()
con.execute(f"CREATE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{token}');")

rel = "hf://datasets/FlyRank/internship-warehouse"

# 3. Verified Step 1 Query
verify_query = f"""
SELECT
    COUNT(*) as total_daily_raw_rows,
    COUNT(DISTINCT content_hash_id) as unique_content_rows_grain,
    MIN(report_date) as slice_start_date,
    MAX(report_date) as slice_end_date
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
"""

# Execute and Display Result
df_verification = con.sql(verify_query).df()

print("=" * 60)
print("VERIFICATION RESULT: UNIT OF ANALYSIS & TIME WINDOW")
print("=" * 60)
print(df_verification)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

VERIFICATION RESULT: UNIT OF ANALYSIS & TIME WINDOW
   total_daily_raw_rows  unique_content_rows_grain slice_start_date  \
0               9841378                     331437       2026-03-01   

  slice_end_date  
0     2026-03-31  


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Step 2: Field Sorting (Feature / Label / Context / Excluded)

Below is the classification of all warehouse fields into the four required buckets for **Lane 2: Content Refresh & Decay Predictor** during decision moment $T_0$ (`month = '2026-03'`):

| Bucket | Field Name | Description & Role |
| :--- | :--- | :--- |
| **1. Feature** | `gsc_clicks` (Aggregated) | Total organic clicks generated in the 30-day window ($T_0$). |
| **1. Feature** | `gsc_impressions` (Aggregated) | Total search impressions received in the 30-day window ($T_0$). |
| **1. Feature** | `gsc_avg_position` (Aggregated) | Average Google SERP ranking position recorded prior to $T_0$. |
| **1. Feature** | `ga4_total_engagement_sec` | Total user engagement duration (seconds) logged before $T_0$. |
| **1. Feature** | `sessions_organic` (Aggregated) | Count of organic traffic sessions logged up to cutoff $T_0$. |
| **2. Label** | `target_decay_flag` | Binary target derivative (`1` if future traffic drops below baseline, `0` otherwise). |
| **2. Label** | `decay_risk_score` | Continuous proxy score calculated to rank pages urgently requiring content refresh. |
| **3. Context** | `content_hash_id` | Primary Key / Grain identifier (Pseudonymized Content Item). |
| **3. Context** | `client_hash_id` | Foreign Key / Grouping identifier (Pseudonymized Client Account). |
| **3. Context** | `report_date` / `month` | Temporal metadata defining the 30-day panel window ($T_0 = \text{'2026-03'}$). |
| **4. Excluded** | `month = '2026-04'` to `'2026-06'` | **WHY:** Post-cutoff performance data. Including future metrics creates severe **Data Leakage**, causing artificial $100\%$ accuracy during training. |
| **4. Excluded** | `ai_chatgpt`, `ai_perplexity`, etc. | **WHY:** Sparse AI-referral metrics in this panel; excluded to maintain a high signal-to-noise ratio and avoid zero-variance noise. |
| **4. Excluded** | `client_has_gsc`, `client_has_ga4` | **WHY:** Static boolean flags that do not vary per page; filtered out during the data availability check step (`IS TRUE`). |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Step 3: Verification with Queries (Proof of Claims)

To ensure zero guesswork in our Data Contract, we verify all four core data claims using explicit DuckDB queries on our mid-panel month (`month=2026-03`):

1. **Grain Proof:** Confirming that aggregating by `content_hash_id` produces exactly 1 row per unique pseudonymized web page.
2. **Row Counts & Window Span:** Verifying the total raw log rows, aggregated slice row count, and the exact date range (`MIN` to `MAX` dates).
3. **Availability & Missing Values:** Filtering active pages using `gsc_data_available = TRUE` and auditing NULL/missing values across our required performance metrics.

In [ ]:
import duckdb
from huggingface_hub import get_token

# 1. Setup DuckDB & Secret
token = get_token()
con = duckdb.connect()
con.execute(f"CREATE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{token}');")

rel = "hf://datasets/FlyRank/internship-warehouse"

# -------------------------------------------------------------
# QUERY 1: GRAIN, ROW COUNTS & DATE WINDOW PROOF
# -------------------------------------------------------------
query_contract_proof = f"""
SELECT
    -- Claim 1: Daily Raw Row Count
    COUNT(*) as total_daily_raw_rows,

    -- Claim 2: Aggregated Grain (Unique Pages)
    COUNT(DISTINCT content_hash_id) as aggregated_grain_rows,

    -- Claim 3: Date Window Span
    MIN(report_date) as window_start_date,
    MAX(report_date) as window_end_date
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
"""

df_proof = con.sql(query_contract_proof).df()

print("=" * 65)
print("PROOF 1: GRAIN, ROW COUNTS & TIME WINDOW")
print("=" * 65)
print(df_proof.to_string(index=False))


# -------------------------------------------------------------
# QUERY 2: AVAILABILITY & MISSING VALUE AUDIT (IS TRUE / NULL CHECK)
# -------------------------------------------------------------
query_availability_audit = f"""
SELECT
    -- Claim 4: Availability Filter Count
    COUNT(DISTINCT content_hash_id) as total_pages,
    COUNT(DISTINCT CASE WHEN gsc_data_available = TRUE THEN content_hash_id END) as surviving_gsc_pages,

    -- Missing Value Audit across candidate features
    SUM(CASE WHEN gsc_clicks IS NULL THEN 1 ELSE 0 END) as null_clicks_count,
    SUM(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END) as null_impressions_count,
    SUM(CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END) as null_position_count,
    SUM(CASE WHEN ga4_total_engagement_sec IS NULL THEN 1 ELSE 0 END) as null_ga4_engagement_count
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
"""

df_audit = con.sql(query_availability_audit).df()

print("\n" + "=" * 65)
print("PROOF 2: AVAILABILITY FILTER & MISSING VALUES AUDIT")
print("=" * 65)
print(df_audit.to_string(index=False))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

PROOF 1: GRAIN, ROW COUNTS & TIME WINDOW
 total_daily_raw_rows  aggregated_grain_rows window_start_date window_end_date
              9841378                 331437        2026-03-01      2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


PROOF 2: AVAILABILITY FILTER & MISSING VALUES AUDIT
 total_pages  surviving_gsc_pages  null_clicks_count  null_impressions_count  null_position_count  null_ga4_engagement_count
      331437               176738                0.0                     0.0            6230317.0                  3018741.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Step 4: Data Limits — What This Data Can Never Tell You

Even with ~81.8M warehouse rows, there are fundamental structural limits to what this dataset can answer. We explicitly document three core data boundaries to avoid erroneous modeling assumptions:

1. **Unbalanced History (Per-Client Depth Variance):**
   * **Limit:** Different clients entered the tracking system at different times (as defined by `dim_clients.gsc_data_start` and `ga4_data_start`).
   * **Consequence:** You cannot assume every page has a full historical timeline. Missing past data for a client does *not* mean zero traffic—it means tracking was not yet active.

2. **GSC-Only Early Rows (Missing GA4 Behavioral Signals):**
   * **Limit:** Google Search Console (GSC) ranking/click data often precedes GA4 analytics integration for many domains.
   * **Consequence:** Early historical records can show high GSC impressions/clicks while `ga4_total_engagement_sec` or `ga4_sessions` are zero or NULL. We cannot infer user on-page behavior during these GSC-only periods.

3. **Window Overlaps & Autocorrelation:**
   * **Limit:** Rolling features computed across overlapping time windows (e.g., March 1–30 vs March 15–April 15) share identical daily log points.
   * **Consequence:** Neighboring windows are heavily autocorrelated. Standard random cross-validation split across overlapping windows causes subtle temporal data leakage.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.